In [13]:
import re
import string
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import nltk


In [14]:

nltk.download('punkt_tab')    # for tokenization
nltk.download('stopwords')    # for stopwords removal


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [15]:

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

## **Clean and normalize text for ML.**
Makes text more uniform, reduces noise, and simplifies vocabulary for model training.

In [16]:
def preprocess(text):
    text = text.lower()  # Lowercase
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    text = re.sub(r'\d+', '', text)  # Remove numbers
    tokens = text.split()

    negation_words = {"not", "no", "nor", "n't"}
    tokens = [word for word in tokens if word not in stop_words or word in negation_words]

    # tokens = [word for word in tokens if word not in stop_words]  # Remove stopwords

    tokens = [stemmer.stem(word) for word in tokens]  # Stemming
    return ' '.join(tokens)

In [17]:
from google.colab import drive
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:


# csv_path = '/content/drive/My Drive/NLP/Restaurant_Reviews.tsv'


# .tsv to csv
# with open(csv_path, 'r', newline='') as tsvfile, open('/content/drive/My Drive/NLP/Restaurant_Reviews.csv', 'w', newline='') as csvfile:
#     tsv_reader = csv.reader(tsvfile, delimiter='\t')
#     csv_writer = csv.writer(csvfile, delimiter=',')
#     for row in tsv_reader:
#         csv_writer.writerow(row)


In [19]:
import csv

# df = pd.DataFrame(data)
df = pd.read_csv("/content/drive/My Drive/NLP/emails.csv")
df

,Text,Spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1
...,...,...
5723,Subject: re : research and development charges...,0
5724,"Subject: re : receipts from visit jim , than...",0
5725,Subject: re : enron case study update wow ! a...,0
5726,"Subject: re : interest david , please , call...",0


In [20]:
df['clean_Text'] = df['Text'].apply(preprocess)
df

,Text,Spam,clean_Text
0,Subject: naturally irresistible your corporate...,1,subject natur irresist corpor ident lt realli ...
1,Subject: the stock trading gunslinger fanny i...,1,subject stock trade gunsling fanni merril muzo...
2,Subject: unbelievable new homes made easy im ...,1,subject unbeliev new home made easi im want sh...
3,Subject: 4 color printing special request add...,1,subject color print special request addit info...
4,"Subject: do not have money , get software cds ...",1,subject not money get softwar cd softwar compa...
...,...,...,...
5723,Subject: re : research and development charges...,0,subject research develop charg gpg forward shi...
5724,"Subject: re : receipts from visit jim , than...",0,subject receipt visit jim thank invit visit ls...
5725,Subject: re : enron case study update wow ! a...,0,subject enron case studi updat wow day super t...
5726,"Subject: re : interest david , please , call...",0,subject interest david pleas call shirley cren...


# **TF-IDF vectorisation**
converts text into numerical vectors

In [21]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['clean_Text'])
y = df['Spam']

In [22]:
y

,Spam
0,1
1,1
2,1
3,1
4,1
...,...
5723,0
5724,0
5725,0
5726,0


In [23]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 478222 stored elements and shape (5728, 25640)>

# **Logistic Regression Training and Predict**

In [24]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# ========== 6. Train Model ==========
model = LogisticRegression()
model.fit(X_train, y_train)

# ========== 7. Evaluate ==========
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# ========== 8. Test on New Sentences ==========
def predict_email(text):
    clean = preprocess(text)
    vec = vectorizer.transform([clean])
    result = model.predict(vec)[0]
    return "Spam" if result == 1 else "Ham"
    # return result


# Try it

print("\nSample Test 1:")
example_text_1 = "Congratulations! You've won a $1000 Walmart gift card. Click the link to claim now."
print("Input:", example_text_1)
print("Predicted Email:", predict_email(example_text_1))

print('')

print("\nSample Test 2:")
example_text_2 = "Hey John, just wanted to check if you're coming to the meeting tomorrow at 10 am."
print("Input:", example_text_2)
print("Predicted Email:", predict_email(example_text_2))


Accuracy: 0.9674229203025014

Classification Report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98      1278
           1       0.99      0.88      0.93       441

    accuracy                           0.97      1719
   macro avg       0.98      0.94      0.96      1719
weighted avg       0.97      0.97      0.97      1719


Sample Test 1:
Input: Congratulations! You've won a $1000 Walmart gift card. Click the link to claim now.
Predicted Email: Spam


Sample Test 2:
Input: Hey John, just wanted to check if you're coming to the meeting tomorrow at 10 am.
Predicted Email: Ham
